# EEG preprocessing and feature extraction pipeline

This notebook preprocesses EEG recordings exported from Bitbrain, segments the signal according to stimulus timestamps, computes power spectral density (PSD), extracts band-power features, and saves filtered data, features, and figures.

The code has been cleaned to remove duplicated installation cells, execution errors, and Spanish variable names/messages where possible. It is intended as supplementary/reproducibility material for the manuscript.

## Expected folder structure

```text
data/
  INPUT/
    <participant_id>/
      eeg_eeg.csv
      stimulus.csv
  OUTPUT/
```

- `eeg_eeg.csv`: raw EEG export without headers.
- `stimulus.csv`: event timestamps in microseconds.

The notebook assumes a 256 Hz sampling rate and the 12-channel Bitbrain montage used in the study.

In [ ]:
# Optional installation cell. Run only if the environment is not already configured.
# %pip install pandas numpy matplotlib scipy mne mne-icalabel onnxruntime

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import simpson

import mne
from mne.preprocessing import ICA

try:
    from mne_icalabel import label_components
    IC_LABEL_AVAILABLE = True
except Exception:
    label_components = None
    IC_LABEL_AVAILABLE = False

mne.set_log_level("WARNING")
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
# -----------------------------
# Configuration
# -----------------------------

INPUT_PATH = Path("data/INPUT")
OUTPUT_PATH = Path("data/OUTPUT")
OUTPUT_FILE_FILTERED = "filtered_eeg.csv"
OUTPUT_FILE_POWER = "band_power_features.csv"

SAMPLING_FREQUENCY = 256

ELECTRODES = ["AF7", "Fp1", "Fp2", "AF8", "F3", "F4", "P3", "P4", "PO7", "O1", "O2", "PO8"]
BITBRAIN_COLUMNS = ELECTRODES + ["sequenceNumber", "timestampReception", "timestamp"]

FREQUENCY_BANDS = {
    "Delta": (1, 4),
    "Theta": (4, 8),
    "Alpha": (8, 13),
    "Beta": (13, 30),
    "Low_Gamma": (30, 40),
}

FINAL_BANDPASS = (1.0, 40.0)
ICA_BANDPASS = (1.0, 100.0)
POWER_LINE_FREQUENCY = 50

# Labels exported by the experimental protocol. They are kept here because they are part of the raw data structure.
# Only labels starting with "image_" are used for stimulus-related EEG epochs.
EVENT_LABELS = [
    "atention_1", "atention_2", "image_1", "descanso_1",
    "atention_3", "image_2", "descanso_2",
    "atention_4", "image_3", "descanso_3",
    "atention_5", "image_4", "descanso_4",
    "atention_6", "image_5",
]

In [ ]:
def load_eeg_and_events(eeg_file: Path, stimulus_file: Path) -> tuple[pd.DataFrame, np.ndarray]:
    """Load raw EEG data and stimulus timestamps.

    Parameters
    ----------
    eeg_file : Path
        CSV file containing the raw EEG recording exported from Bitbrain.
    stimulus_file : Path
        CSV file containing event timestamps in microseconds.

    Returns
    -------
    eeg_df : pandas.DataFrame
        EEG dataframe with electrode and timestamp columns.
    event_times_seconds : numpy.ndarray
        Event times in seconds, aligned to the first EEG timestamp.
    """
    eeg_df = pd.read_csv(eeg_file, header=None).dropna()
    eeg_df.columns = BITBRAIN_COLUMNS

    event_times_us = pd.read_csv(stimulus_file, header=None).iloc[:, 0].to_numpy()
    event_times_seconds = (event_times_us - eeg_df["timestampReception"].iloc[0]) / 1e6

    return eeg_df, event_times_seconds


def create_mne_raw(eeg_df: pd.DataFrame, sfreq: int = SAMPLING_FREQUENCY) -> mne.io.RawArray:
    """Create an MNE RawArray from Bitbrain EEG data."""
    eeg_data_volts = eeg_df[ELECTRODES].to_numpy().T * 1e-6  # microvolts to volts
    info = mne.create_info(ch_names=ELECTRODES, sfreq=sfreq, ch_types="eeg")
    raw = mne.io.RawArray(eeg_data_volts, info, verbose=False)

    montage = mne.channels.make_standard_montage("standard_1005")
    raw.set_montage(montage, on_missing="ignore")
    return raw

In [ ]:
def preprocess_eeg(raw: mne.io.RawArray, use_iclabel: bool = True) -> mne.io.RawArray:
    """Preprocess EEG data using filtering, average reference, ICA, and optional ICLabel.

    The function uses an initial broad band-pass filter for ICA, applies average reference,
    fits extended Infomax ICA, optionally excludes non-brain components using ICLabel,
    applies a notch filter for line noise, and finally applies a 1-40 Hz band-pass filter.

    If ICLabel or one of its optional backends is unavailable, ICA is fitted but no automatic
    component exclusion is applied. This prevents the notebook from failing because of a
    missing optional dependency.
    """
    nyquist = raw.info["sfreq"] / 2
    ica_high = min(ICA_BANDPASS[1], nyquist - 1)
    final_high = min(FINAL_BANDPASS[1], nyquist - 1)

    filtered_for_ica = raw.copy().filter(l_freq=ICA_BANDPASS[0], h_freq=ica_high, verbose=False)
    filtered_for_ica.set_eeg_reference("average", verbose=False)

    ica = ICA(
        n_components=len(ELECTRODES) - 1,
        random_state=97,
        method="infomax",
        fit_params={"extended": True},
        max_iter="auto",
    )
    ica.fit(filtered_for_ica, verbose=False)

    exclude_idx = []
    if use_iclabel and IC_LABEL_AVAILABLE:
        try:
            ic_labels = label_components(filtered_for_ica, ica, method="iclabel")
            labels = ic_labels["labels"]
            exclude_idx = [idx for idx, label in enumerate(labels) if label not in ["brain", "other"]]
            print(f"Automatically excluded ICA components: {exclude_idx}")
        except Exception as exc:
            print(f"ICLabel was not applied: {exc}")
            print("No ICA components were automatically excluded.")
    else:
        print("ICLabel is unavailable or disabled. No ICA components were automatically excluded.")

    cleaned = raw.copy()
    ica.apply(cleaned, exclude=exclude_idx, verbose=False)
    cleaned.notch_filter(freqs=POWER_LINE_FREQUENCY, verbose=False)
    cleaned.filter(l_freq=FINAL_BANDPASS[0], h_freq=final_high, verbose=False)

    return cleaned

In [ ]:
def extract_stimulus_epochs(
    raw: mne.io.RawArray,
    event_times: np.ndarray,
    labels: list[str] = EVENT_LABELS,
    baseline_duration: float = 0.2,
) -> dict[str, list[mne.io.RawArray]]:
    """Extract stimulus-related EEG segments and apply baseline correction.

    Only labels beginning with "image_" are extracted, because these correspond to the
    emotional stimulus presentation events in the protocol.
    """
    onsets = event_times[:-1]
    durations = np.diff(event_times)

    epochs = {}
    for onset, duration, label in zip(onsets, durations, labels):
        if not label.startswith("image_"):
            continue

        tmin = max(onset, raw.times[0])
        tmax = min(onset + duration, raw.times[-1])
        if tmax <= tmin:
            continue

        epoch = raw.copy().crop(tmin=tmin, tmax=tmax)

        baseline_start = max(onset - baseline_duration, raw.times[0])
        baseline_end = max(min(onset, raw.times[-1]), baseline_start)
        if baseline_end > baseline_start:
            baseline = raw.copy().crop(tmin=baseline_start, tmax=baseline_end)
            baseline_mean = baseline.get_data().mean(axis=1, keepdims=True)
            epoch_data = epoch.get_data() - baseline_mean
            epoch = mne.io.RawArray(epoch_data, epoch.info.copy(), verbose=False)

        epochs.setdefault(label, []).append(epoch)

    return epochs

In [ ]:
def compute_band_power(raw: mne.io.RawArray, bands: dict[str, tuple[float, float]] = FREQUENCY_BANDS):
    """Compute PSD using Welch's method and integrate power within frequency bands.

    Returns
    -------
    band_powers : dict
        Band power per EEG channel in microvolts squared (µV²).
    psd : mne.time_frequency.Spectrum
        MNE PSD object.
    """
    n_per_seg = max(raw.n_times // 3, 8)
    n_overlap = n_per_seg // 2

    psd = raw.compute_psd(
        method="welch",
        fmin=FINAL_BANDPASS[0],
        fmax=FINAL_BANDPASS[1],
        n_per_seg=n_per_seg,
        n_overlap=n_overlap,
        verbose=False,
    )

    psd_data = psd.get_data()  # V²/Hz
    freqs = psd.freqs
    frequency_resolution = freqs[1] - freqs[0] if len(freqs) > 1 else 1.0

    band_powers = {}
    for band, (fmin, fmax) in bands.items():
        mask = (freqs >= fmin) & (freqs < fmax)
        if not mask.any():
            band_powers[band] = np.full(len(raw.ch_names), np.nan)
            continue
        power = simpson(psd_data[:, mask], dx=frequency_resolution, axis=1)
        band_powers[band] = power * 1e12  # V² to µV²

    return band_powers, psd

In [ ]:
def plot_eeg_results(raw: mne.io.RawArray, band_powers: dict, psd, stimulus_label: str, output_folder: Path) -> None:
    """Generate and save topographic, PSD, and band-power distribution plots."""
    output_folder.mkdir(parents=True, exist_ok=True)

    # Topographic maps
    fig, axes = plt.subplots(1, len(band_powers), figsize=(4 * len(band_powers), 4))
    if len(band_powers) == 1:
        axes = [axes]

    for ax, (band, power) in zip(axes, band_powers.items()):
        im, _ = mne.viz.plot_topomap(
            power,
            pos=raw.info,
            axes=ax,
            show=False,
            names=raw.ch_names,
        )
        ax.set_title(band)
        fig.colorbar(im, ax=ax, format="%.2f", label="Power (µV²)")

    fig.suptitle(f"Band-power topography: {stimulus_label}")
    fig.tight_layout()
    fig.savefig(output_folder / f"{stimulus_label}_topomap.png", dpi=300)
    plt.close(fig)

    # PSD plot
    psd_fig = psd.plot(average=False, amplitude=False, picks="eeg", xscale="linear", dB=False, show=False)
    psd_fig.suptitle(f"Power spectral density: {stimulus_label}")
    psd_fig.savefig(output_folder / f"{stimulus_label}_psd.png", dpi=300)
    plt.close(psd_fig)

    # Band-power bar plot
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(raw.ch_names))
    width = 0.8 / max(len(band_powers), 1)

    for i, (band, power) in enumerate(band_powers.items()):
        ax.bar(x + i * width, power, width, label=band)

    ax.set_xticks(x + width * (len(band_powers) - 1) / 2)
    ax.set_xticklabels(raw.ch_names, rotation=45)
    ax.set_ylabel("Power (µV²)")
    ax.set_title(f"Band-power distribution: {stimulus_label}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_folder / f"{stimulus_label}_band_power.png", dpi=300)
    plt.close(fig)

In [ ]:
def save_participant_results(
    cleaned_raw: mne.io.RawArray,
    participant_band_powers: dict[str, dict[str, np.ndarray]],
    filtered_output_file: Path,
    power_output_file: Path,
) -> None:
    """Save filtered EEG and band-power features to CSV files."""
    filtered_data_microvolts = cleaned_raw.get_data().T * 1e6
    pd.DataFrame(filtered_data_microvolts, columns=cleaned_raw.ch_names).to_csv(filtered_output_file, index=False)

    features = {}
    for stimulus_label, band_dict in participant_band_powers.items():
        for band, power_values in band_dict.items():
            for channel, value in zip(cleaned_raw.ch_names, power_values):
                features[f"{stimulus_label}_{band}_{channel}"] = value

    pd.DataFrame([features]).to_csv(power_output_file, index=False)

In [ ]:
def process_participant(participant_folder: Path, output_root: Path, use_iclabel: bool = True) -> None:
    """Run the full EEG preprocessing and feature extraction pipeline for one participant."""
    participant_id = participant_folder.name
    output_folder = output_root / participant_id
    output_folder.mkdir(parents=True, exist_ok=True)

    eeg_file = participant_folder / "eeg_eeg.csv"
    stimulus_file = participant_folder / "stimulus.csv"

    if not eeg_file.exists() or not stimulus_file.exists():
        print(f"Skipping {participant_id}: missing eeg_eeg.csv or stimulus.csv")
        return

    print(f"Processing participant: {participant_id}")

    eeg_df, event_times = load_eeg_and_events(eeg_file, stimulus_file)
    raw = create_mne_raw(eeg_df)
    cleaned_raw = preprocess_eeg(raw, use_iclabel=use_iclabel)
    stimulus_epochs = extract_stimulus_epochs(cleaned_raw, event_times)

    participant_results = {}
    for stimulus_label, epoch_list in stimulus_epochs.items():
        if not epoch_list:
            continue
        band_powers, psd = compute_band_power(epoch_list[0])
        participant_results[stimulus_label] = band_powers
        plot_eeg_results(epoch_list[0], band_powers, psd, stimulus_label, output_folder)

    save_participant_results(
        cleaned_raw,
        participant_results,
        output_folder / OUTPUT_FILE_FILTERED,
        output_folder / OUTPUT_FILE_POWER,
    )
    print(f"Saved results for participant {participant_id} in {output_folder}")

In [ ]:
# Run the full pipeline for all participants.
# Set use_iclabel=False if ICLabel optional dependencies are not installed.

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

for participant_folder in sorted(INPUT_PATH.iterdir()):
    if participant_folder.is_dir():
        process_participant(participant_folder, OUTPUT_PATH, use_iclabel=True)

## Notes for reproducibility

- PSD is estimated using Welch's method through MNE-Python.
- Band power is integrated over canonical EEG frequency bands and converted from V² to µV².
- ICA component labelling uses `mne-icalabel` when available. If its optional backends (`onnxruntime` or `torch`) are not installed, the pipeline does not automatically exclude ICA components.
- The event labels are retained in their original exported form because they correspond to the acquisition protocol.